# BT4222 Group 1 — JDSearch Personalised Re-ranking

### NingJing — Query-Match & Popularity Features, Model Evaluation Input

This notebook covers:
1. Query-match and popularity features
2. Baseline models (NMF, NCF) needed as comparison inputs for the bias diagnosis

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import gc
import glob
import pickle
from pathlib import Path
from collections import Counter, defaultdict
import matplotlib.pyplot as plt

BASE = Path("/content/drive/MyDrive/BT4222 Final/JDSearch/Raw Data")
USER_TXT = BASE / "user_behavior_data.txt"
PRODUCT_TXT = BASE / "product_meta_data.txt"
CHUNKS_DIR = BASE / "product_chunks"
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

# New: dedicated folder for saved model artifacts (persists on Drive across Colab sessions/crashes)
ARTIFACTS_DIR = BASE.parent / "model_artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = BASE.parent / "processed"


Mounted at /content/drive


## Load `user_df`

In [2]:
user_df = pd.read_csv(USER_TXT, sep="\t", dtype=str, low_memory=False)
print(user_df.shape)


(173831, 7)


## Build `candidate_interactions_df`

In [3]:
def explode_with_position(df, id_col, label_col):
    rows = []
    for idx, row in df[[id_col, label_col]].dropna().iterrows():
        ids = row[id_col].split('_')
        labels = row[label_col].split('_')
        for pos, (i, l) in enumerate(zip(ids, labels)):
            rows.append((idx, pos, i, float(l)))
    return pd.DataFrame(rows, columns=['session_id', 'position', 'item_id', 'label'])

chunk_size = 10000
all_chunks = []
for start in range(0, len(user_df), chunk_size):
    chunk = user_df.iloc[start:start + chunk_size]
    result = explode_with_position(chunk, 'candidate_wid_list', 'candidate_label_list')
    all_chunks.append(result)
    del chunk, result
    gc.collect()
    print(f"Processed up to row {start}...")

candidate_interactions_df = pd.concat(all_chunks, ignore_index=True)
del all_chunks
gc.collect()
print(candidate_interactions_df.shape)


Processed up to row 0...
Processed up to row 10000...
Processed up to row 20000...
Processed up to row 30000...
Processed up to row 40000...
Processed up to row 50000...
Processed up to row 60000...
Processed up to row 70000...
Processed up to row 80000...
Processed up to row 90000...
Processed up to row 100000...
Processed up to row 110000...
Processed up to row 120000...
Processed up to row 130000...
Processed up to row 140000...
Processed up to row 150000...
Processed up to row 160000...
Processed up to row 170000...
(15510012, 4)


## Merge product metadata (brand / shop / category IDs)

In [4]:
# Build the parquet chunk cache once, if it doesn't already exist
existing_chunks = sorted(glob.glob(str(CHUNKS_DIR / "*.parquet")))
if not existing_chunks:
    print("No cached chunks found — building them from the raw text file (one-off, ~slow)...")
    chunksize = 500_000
    reader = pd.read_csv(PRODUCT_TXT, sep='\t', dtype=str, chunksize=chunksize)
    for i, chunk in enumerate(reader):
        out_path = CHUNKS_DIR / f"chunk_{i:04d}.parquet"
        chunk.to_parquet(out_path, index=False)
        print(f"Saved {out_path.name} ({len(chunk)} rows)")
    print("Done writing all chunks.")
else:
    print(f"Found {len(existing_chunks)} cached chunk files, skipping rebuild.")

chunk_files = sorted(glob.glob(str(CHUNKS_DIR / "*.parquet")))


Found 25 cached chunk files, skipping rebuild.


In [5]:
needed_item_ids = set(candidate_interactions_df['item_id'].unique())
print(f"Unique candidate items needing content features: {len(needed_item_ids):,}")

cols_needed = ['wid', 'brand_id', 'shop_id', 'cate_id_1', 'cate_id_2', 'cate_id_3', 'cate_id_4']

product_rows = []
for f in chunk_files:
    chunk = pd.read_parquet(f, columns=cols_needed)
    matched = chunk[chunk['wid'].isin(needed_item_ids)]
    product_rows.append(matched)
    del chunk
    gc.collect()

product_lookup_df = pd.concat(product_rows, ignore_index=True)
del product_rows
gc.collect()
print(product_lookup_df.shape)


Unique candidate items needing content features: 8,362,535
(8305622, 7)


In [6]:
candidate_interactions_df = candidate_interactions_df.merge(
    product_lookup_df, left_on='item_id', right_on='wid', how='left'
)
candidate_interactions_df.drop(columns=['wid'], inplace=True)

print(candidate_interactions_df.shape)
print(candidate_interactions_df[['brand_id', 'shop_id', 'cate_id_1']].isna().mean())

del product_lookup_df
gc.collect()


(15510012, 10)
brand_id     0.004535
shop_id      0.004535
cate_id_1    0.004535
dtype: float64


0

## Load the team's train/val/test split

In [ ]:
# Load the train/val/test split from step_1_data_preprocessing.ipynb and merge onto our existing candidate table
split_df = pd.read_csv(OUTPUT_DIR / "user_split.csv")  # columns: session_id, split
print(split_df.shape)
print(split_df['split'].value_counts())

candidate_interactions_df = candidate_interactions_df.merge(
    split_df, on='session_id', how='left'
)
print(candidate_interactions_df['split'].isna().sum(), "rows with no split assigned")
print(candidate_interactions_df['split'].value_counts())

(171175, 2)
split
train    136940
test      17118
val       17117
Name: count, dtype: int64
161963 rows with no split assigned
split
train    12279873
val       1536651
test      1531525
Name: count, dtype: int64


In [8]:
# Drop rows belonging to the excluded sessions (no positive or no negative label)
candidate_interactions_df = candidate_interactions_df[candidate_interactions_df['split'].notna()].copy()
print(candidate_interactions_df.shape)
print(candidate_interactions_df['split'].value_counts())
gc.collect()

(15348049, 11)
split
train    12279873
val       1536651
test      1531525
Name: count, dtype: int64


0

## Feature 1: Query-match features

Does the candidate product's name / brand / category match a term in the search query.

In [9]:
text_cols_needed = ['wid', 'name', 'brand_name', 'cate_name_1', 'cate_name_2', 'cate_name_3', 'cate_name_4']

text_rows = []
for f in chunk_files:
    chunk = pd.read_parquet(f, columns=text_cols_needed)
    matched = chunk[chunk['wid'].isin(needed_item_ids)]
    text_rows.append(matched)
    del chunk
    gc.collect()

product_text_df = pd.concat(text_rows, ignore_index=True)
del text_rows
gc.collect()
print(product_text_df.shape)

candidate_interactions_df = candidate_interactions_df.merge(
    product_text_df, left_on='item_id', right_on='wid', how='left'
)
candidate_interactions_df.drop(columns=['wid'], inplace=True)
print(candidate_interactions_df.shape)
del product_text_df
gc.collect()

(8305622, 7)
(15348049, 17)


0

In [10]:
def parse_query_terms(q):
    if pd.isna(q) or q == '':
        return []
    return list(set(q.split('\x18')))

user_df['query_terms'] = user_df['query'].apply(parse_query_terms)

query_long = user_df[['query_terms']].explode('query_terms').reset_index()
query_long.columns = ['session_id', 'term']
query_long = query_long.dropna(subset=['term']).drop_duplicates()
print(query_long.shape)

(473743, 2)


In [11]:
match_fields = ['name', 'brand_name', 'cate_name_1', 'cate_name_2', 'cate_name_3', 'cate_name_4']

for field in match_fields:
    tmp = candidate_interactions_df[['session_id', field]].rename(columns={field: 'term'})
    merged = tmp.merge(query_long, on=['session_id', 'term'], how='left', indicator=True)
    candidate_interactions_df[f'{field}_query_match'] = (merged['_merge'].values == 'both')
    del tmp, merged
    gc.collect()
    print(f"{field}: match rate = {candidate_interactions_df[f'{field}_query_match'].mean():.4f}")

match_cols = [f'{f}_query_match' for f in match_fields]
candidate_interactions_df['query_match_count'] = candidate_interactions_df[match_cols].sum(axis=1)
print(candidate_interactions_df['query_match_count'].value_counts())

name: match rate = 0.0000
brand_name: match rate = 0.0454
cate_name_1: match rate = 0.0015
cate_name_2: match rate = 0.0272
cate_name_3: match rate = 0.1610
cate_name_4: match rate = 0.1508
query_match_count
0    11771169
2     1864427
1     1475402
3      227942
4        9109
Name: count, dtype: int64


In [12]:
SAVE_PATH = OUTPUT_DIR / "ningjing_candidate_features_checkpoint.parquet"
candidate_interactions_df.to_parquet(SAVE_PATH, index=False)
print(f"Saved checkpoint to {SAVE_PATH}, shape: {candidate_interactions_df.shape}")

Saved checkpoint to /content/drive/MyDrive/BT4222 Final/JDSearch/processed/ningjing_candidate_features_checkpoint.parquet, shape: (15348049, 24)


## Feature 2: Popularity features (training split only)

Item, brand, and category popularity, computed using training-split sessions only, so the feature does not leak information from the validation/test sessions.

In [13]:
train_mask = candidate_interactions_df['split'] == 'train'

item_pop_train = (
    candidate_interactions_df[train_mask]
    .assign(engaged=lambda d: d['label'] > 0)
    .groupby('item_id')['engaged']
    .sum()
)

brand_pop_train = (
    candidate_interactions_df[train_mask]
    .assign(engaged=lambda d: d['label'] > 0)
    .groupby('brand_id')['engaged']
    .sum()
)

cate4_pop_train = (
    candidate_interactions_df[train_mask]
    .assign(engaged=lambda d: d['label'] > 0)
    .groupby('cate_id_4')['engaged']
    .sum()
)

print(item_pop_train.describe())
print(brand_pop_train.describe())
print(cate4_pop_train.describe())

count    7.021061e+06
mean     3.652539e-02
std      4.029227e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      3.090000e+02
Name: engaged, dtype: float64
count    160507.000000
mean          1.593937
std          30.443649
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max        6995.000000
Name: engaged, dtype: float64
count     6732.000000
mean        38.003268
std        187.759262
min          0.000000
25%          0.000000
50%          3.000000
75%         22.000000
max      11785.000000
Name: engaged, dtype: float64


In [14]:
candidate_interactions_df['item_pop_train'] = candidate_interactions_df['item_id'].map(item_pop_train).fillna(0)
candidate_interactions_df['brand_pop_train'] = candidate_interactions_df['brand_id'].map(brand_pop_train).fillna(0)
candidate_interactions_df['cate4_pop_train'] = candidate_interactions_df['cate_id_4'].map(cate4_pop_train).fillna(0)

print(candidate_interactions_df[['item_pop_train','brand_pop_train','cate4_pop_train']].describe())

       item_pop_train  brand_pop_train  cate4_pop_train
count    1.534805e+07     1.534805e+07     1.534805e+07
mean     4.806952e-01     6.703111e+02     8.787385e+02
std      5.091341e+00     1.546430e+03     2.285130e+03
min      0.000000e+00     0.000000e+00     0.000000e+00
25%      0.000000e+00     3.000000e+00     7.400000e+01
50%      0.000000e+00     2.300000e+01     2.230000e+02
75%      0.000000e+00     2.590000e+02     6.090000e+02
max      3.090000e+02     6.995000e+03     1.178500e+04


Save checkpoint again after adding the popularity features.

In [15]:
candidate_interactions_df.to_parquet(OUTPUT_DIR / "ningjing_candidate_features_checkpoint.parquet", index=False)
print("Saved.")

Saved.


## Baseline model: NMF

Needed as one of the comparison models in the bias diagnosis below.

In [16]:
type_weight = {'CLICK': 1.0, 'CART': 3.0, 'ORD': 5.0, 'FLW': 2.0}

session_idx_list = []   # row index per session, 0..N-1
item_id_list = []       # raw item id strings, remapped to column indices later
weight_list = []        # implicit-feedback weight

chunk_size = 10000
session_counter = 0

for start in range(0, len(user_df), chunk_size):
    chunk = user_df.iloc[start:start + chunk_size]
    for row_offset, (_, row) in enumerate(chunk[['history_wid_list', 'history_type_list']].iterrows()):
        session_idx = session_counter + row_offset
        wid_str = row['history_wid_list']
        type_str = row['history_type_list']
        if pd.isna(wid_str) or pd.isna(type_str):
            continue
        wids = wid_str.split('_')
        types = type_str.split('_')
        if len(wids) != len(types):
            continue  # skip malformed rows
        for wid, typ in zip(wids, types):
            w = type_weight.get(typ, 1.0)
            session_idx_list.append(session_idx)
            item_id_list.append(wid)
            weight_list.append(w)
    session_counter += len(chunk)
    del chunk
    gc.collect()
    print(f"Processed up to session {session_counter}...")

print(f"Total (session, item, weight) interaction records: {len(session_idx_list)}")


Processed up to session 10000...
Processed up to session 20000...
Processed up to session 30000...
Processed up to session 40000...
Processed up to session 50000...
Processed up to session 60000...
Processed up to session 70000...
Processed up to session 80000...
Processed up to session 90000...
Processed up to session 100000...
Processed up to session 110000...
Processed up to session 120000...
Processed up to session 130000...
Processed up to session 140000...
Processed up to session 150000...
Processed up to session 160000...
Processed up to session 170000...
Processed up to session 173831...
Total (session, item, weight) interaction records: 26667260


In [17]:
from scipy.sparse import coo_matrix

session_idx_arr = np.array(session_idx_list, dtype=np.int32)

# Factorize item_id strings into integer column indices
item_codes, item_categories = pd.factorize(np.array(item_id_list))
item_codes = item_codes.astype(np.int32)

weight_arr = np.array(weight_list, dtype=np.float32)

n_users = int(session_idx_arr.max()) + 1
n_items = len(item_categories)
print(f"Matrix shape: {n_users} users x {n_items} items")

# Build sparse matrix; duplicate (user, item) entries are automatically summed on conversion to CSR
interaction_matrix = coo_matrix(
    (weight_arr, (session_idx_arr, item_codes)),
    shape=(n_users, n_items)
).tocsr()

print(f"Non-zero entries: {interaction_matrix.nnz}")
print(f"Sparsity: {1 - interaction_matrix.nnz / (float(n_users) * float(n_items)):.8f}")

del session_idx_list, item_id_list, weight_list, session_idx_arr, item_codes, weight_arr
gc.collect()


Matrix shape: 173831 users x 5584543 items
Non-zero entries: 17322100
Sparsity: 0.99998216


30

In [18]:
# Distinct-user count per item (how many different sessions interacted with this item)
item_user_count = np.diff(interaction_matrix.tocsc().indptr)

print(f"Items with >=1 distinct user:  {(item_user_count >= 1).sum()}")
print(f"Items with >=3 distinct users: {(item_user_count >= 3).sum()}")
print(f"Items with >=5 distinct users: {(item_user_count >= 5).sum()}")
print(f"Items with >=10 distinct users: {(item_user_count >= 10).sum()}")


Items with >=1 distinct user:  5584543
Items with >=3 distinct users: 956932
Items with >=5 distinct users: 526137
Items with >=10 distinct users: 243627


In [19]:
MIN_USERS = 5
keep_mask = item_user_count >= MIN_USERS

kept_item_categories = item_categories[keep_mask]
interaction_matrix_filtered = interaction_matrix.tocsc()[:, keep_mask].tocsr()

print(f"Filtered matrix shape: {interaction_matrix_filtered.shape}")
print(f"Non-zero entries: {interaction_matrix_filtered.nnz}")

# Mapping from original item_id string -> column index, needed to score candidates later
item_id_to_col = {item_id: idx for idx, item_id in enumerate(kept_item_categories)}
print(f"Items retained for NMF/NCF training: {len(item_id_to_col)}")

# Save to Drive (not /content) so it survives a Colab crash/disconnect
with open(ARTIFACTS_DIR / "nmf_artifacts.pkl", 'wb') as f:
    pickle.dump({'interaction_matrix_filtered': interaction_matrix_filtered,
                 'item_id_to_col': item_id_to_col}, f)
print("Saved nmf_artifacts.pkl to Drive")


Filtered matrix shape: (173831, 526137)
Non-zero entries: 10575422
Items retained for NMF/NCF training: 526137
Saved nmf_artifacts.pkl to Drive


In [20]:
from sklearn.decomposition import NMF
import time

N_COMPONENTS = 32

nmf_model = NMF(
    n_components=N_COMPONENTS,
    init='nndsvda',
    solver='mu',
    max_iter=50,
    random_state=42
)

t0 = time.time()
W = nmf_model.fit_transform(interaction_matrix_filtered)  # user latent factors, shape (n_users, k)
H = nmf_model.components_                                   # item latent factors, shape (k, n_items)
elapsed = time.time() - t0

print(f"Training took {elapsed:.1f} seconds")
print(f"W (user factors) shape: {W.shape}")
print(f"H (item factors) shape: {H.shape}")
print(f"Reconstruction error: {nmf_model.reconstruction_err_:.4f}")

with open(ARTIFACTS_DIR / "nmf_wh.pkl", 'wb') as f:
    pickle.dump({'W': W, 'H': H}, f)
print("Saved nmf_wh.pkl to Drive")


Training took 25.7 seconds
W (user factors) shape: (173831, 32)
H (item factors) shape: (32, 526137)
Reconstruction error: 19835.0801
Saved nmf_wh.pkl to Drive


In [21]:
# Map each candidate's item_id string to its NMF/NCF column index (NaN if item has no trained vector)
candidate_interactions_df['nmf_item_idx'] = candidate_interactions_df['item_id'].map(item_id_to_col)

has_nmf_vector = candidate_interactions_df['nmf_item_idx'].notna()
coverage_pct = has_nmf_vector.mean() * 100

print(f"Candidates WITH a trained item vector:    {has_nmf_vector.sum():,} ({coverage_pct:.1f}%)")
print(f"Candidates WITHOUT a trained item vector: {(~has_nmf_vector).sum():,} ({100-coverage_pct:.1f}%)")


Candidates WITH a trained item vector:    3,355,646 (21.9%)
Candidates WITHOUT a trained item vector: 11,992,403 (78.1%)


In [22]:
# Only compute dot products for candidates that have a valid NMF item vector
matched_mask = candidate_interactions_df['nmf_item_idx'].notna()
matched_df = candidate_interactions_df[matched_mask]

session_idx_arr = matched_df['session_id'].values.astype(np.int64)
item_idx_arr = matched_df['nmf_item_idx'].values.astype(np.int64)

chunk_size = 500_000
nmf_scores = np.empty(len(matched_df), dtype=np.float32)

for start in range(0, len(matched_df), chunk_size):
    end = min(start + chunk_size, len(matched_df))
    u = W[session_idx_arr[start:end]]      # user latent vectors, shape (chunk, 32)
    v = H[:, item_idx_arr[start:end]].T     # item latent vectors, shape (chunk, 32)
    nmf_scores[start:end] = (u * v).sum(axis=1)   # dot product = predicted affinity score
    del u, v
    gc.collect()
    print(f"Scored {end:,} / {len(matched_df):,} matched candidates...")

candidate_interactions_df['nmf_score'] = np.nan
candidate_interactions_df.loc[matched_df.index, 'nmf_score'] = nmf_scores

print(f"NMF scores computed for {len(matched_df):,} candidates")
print(candidate_interactions_df['nmf_score'].describe())

del matched_df, session_idx_arr, item_idx_arr
gc.collect()


Scored 500,000 / 3,355,646 matched candidates...
Scored 1,000,000 / 3,355,646 matched candidates...
Scored 1,500,000 / 3,355,646 matched candidates...
Scored 2,000,000 / 3,355,646 matched candidates...
Scored 2,500,000 / 3,355,646 matched candidates...
Scored 3,000,000 / 3,355,646 matched candidates...
Scored 3,355,646 / 3,355,646 matched candidates...
NMF scores computed for 3,355,646 candidates
count    3.355646e+06
mean     8.904957e-03
std      4.524982e-01
min      0.000000e+00
25%      9.622078e-06
50%      9.665758e-05
75%      7.550529e-04
max      2.768510e+02
Name: nmf_score, dtype: float64


0

Combine the NMF score with a popularity fallback for candidates with no trained NMF vector. The fallback uses `item_pop_train` (training-split only), not a popularity score computed over the full dataset, to avoid leaking validation/test information into the score.

In [23]:
OFFSET = 1e6  # guarantees NMF-scored candidates always rank above the popularity fallback

candidate_interactions_df['hybrid_score'] = np.where(
    candidate_interactions_df['nmf_score'].notna(),
    candidate_interactions_df['nmf_score'] + OFFSET,
    candidate_interactions_df['item_pop_train']
)


## Baseline model: NCF

Same item vocabulary and user-history matrix as NMF, so the two baselines are directly comparable.

In [24]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

N_USERS, N_ITEMS = interaction_matrix_filtered.shape

coo = interaction_matrix_filtered.tocoo()
pos_users = coo.row.astype(np.int64)
pos_items = coo.col.astype(np.int64)
print(f"Number of positive (user, item) pairs: {len(pos_users):,}")


class NCFDataset(Dataset):
    # Implicit feedback: 1 random negative sampled per positive pair
    def __init__(self, pos_users, pos_items, n_items, n_neg=1):
        self.pos_users = pos_users
        self.pos_items = pos_items
        self.n_items = n_items
        self.n_neg = n_neg

    def __len__(self):
        return len(self.pos_users) * (1 + self.n_neg)

    def __getitem__(self, idx):
        pair_idx = idx // (1 + self.n_neg)
        is_negative = idx % (1 + self.n_neg) != 0
        user = self.pos_users[pair_idx]
        if is_negative:
            item = np.random.randint(0, self.n_items)
            label = 0.0
        else:
            item = self.pos_items[pair_idx]
            label = 1.0
        return user, item, label


class NCF(nn.Module):
    # user/item embeddings -> MLP -> interaction probability
    def __init__(self, n_users, n_items, emb_dim=16, hidden_dims=(32, 16)):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        layers, input_dim = [], emb_dim * 2
        for h in hidden_dims:
            layers += [nn.Linear(input_dim, h), nn.ReLU()]
            input_dim = h
        layers.append(nn.Linear(input_dim, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, user, item):
        x = torch.cat([self.user_emb(user), self.item_emb(item)], dim=1)
        return self.mlp(x).squeeze(-1)


model = NCF(N_USERS, N_ITEMS, emb_dim=16).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCEWithLogitsLoss()
loader = DataLoader(NCFDataset(pos_users, pos_items, N_ITEMS, n_neg=1),
                     batch_size=4096, shuffle=True, num_workers=0)
print(f"Training examples this epoch: {len(loader.dataset):,}")

model.train()
t0 = time.time()
total_loss, n_batches = 0.0, 0
for users_b, items_b, labels_b in loader:
    users_b, items_b, labels_b = users_b.to(device), items_b.to(device), labels_b.float().to(device)
    optimizer.zero_grad()
    loss = criterion(model(users_b, items_b), labels_b)
    loss.backward()
    optimizer.step()
    total_loss += loss.item()
    n_batches += 1
    if n_batches % 1000 == 0:
        print(f"Batch {n_batches}, avg loss: {total_loss/n_batches:.4f}, elapsed {time.time()-t0:.1f}s")

print(f"Epoch done. Avg loss: {total_loss/n_batches:.4f}. Time: {time.time()-t0:.1f}s")

# Save to Drive (not /content) so it survives a Colab crash/disconnect
torch.save(model.state_dict(), ARTIFACTS_DIR / "ncf_model.pt")
print("Saved ncf_model.pt to Drive")


Using device: cuda
Number of positive (user, item) pairs: 10,575,422
Training examples this epoch: 21,150,844
Batch 1000, avg loss: 0.6887, elapsed 8.3s
Batch 2000, avg loss: 0.6657, elapsed 14.8s
Batch 3000, avg loss: 0.6474, elapsed 21.4s
Batch 4000, avg loss: 0.6343, elapsed 27.9s
Batch 5000, avg loss: 0.6245, elapsed 34.4s
Epoch done. Avg loss: 0.6231. Time: 36.3s
Saved ncf_model.pt to Drive


In [25]:
model.eval()

matched_mask = candidate_interactions_df['nmf_item_idx'].notna()   # same vocabulary as NMF
matched_df = candidate_interactions_df[matched_mask]

session_idx_arr = matched_df['session_id'].values.astype(np.int64)
item_idx_arr = matched_df['nmf_item_idx'].values.astype(np.int64)

ncf_scores = np.empty(len(matched_df), dtype=np.float32)
batch_size = 200_000
with torch.no_grad():
    for start in range(0, len(matched_df), batch_size):
        end = min(start + batch_size, len(matched_df))
        u = torch.from_numpy(session_idx_arr[start:end]).to(device)
        v = torch.from_numpy(item_idx_arr[start:end]).to(device)
        ncf_scores[start:end] = torch.sigmoid(model(u, v)).cpu().numpy()
        print(f"Scored {end:,} / {len(matched_df):,}")

candidate_interactions_df['ncf_score'] = np.nan
candidate_interactions_df.loc[matched_df.index, 'ncf_score'] = ncf_scores
print(candidate_interactions_df['ncf_score'].describe())

del matched_df, session_idx_arr, item_idx_arr
gc.collect()


Scored 200,000 / 3,355,646
Scored 400,000 / 3,355,646
Scored 600,000 / 3,355,646
Scored 800,000 / 3,355,646
Scored 1,000,000 / 3,355,646
Scored 1,200,000 / 3,355,646
Scored 1,400,000 / 3,355,646
Scored 1,600,000 / 3,355,646
Scored 1,800,000 / 3,355,646
Scored 2,000,000 / 3,355,646
Scored 2,200,000 / 3,355,646
Scored 2,400,000 / 3,355,646
Scored 2,600,000 / 3,355,646
Scored 2,800,000 / 3,355,646
Scored 3,000,000 / 3,355,646
Scored 3,200,000 / 3,355,646
Scored 3,355,646 / 3,355,646
count    3.355646e+06
mean     5.028002e-01
std      2.166549e-01
min      1.030115e-01
25%      3.284378e-01
50%      4.100789e-01
75%      6.709283e-01
max      9.972118e-01
Name: ncf_score, dtype: float64


32

Same fallback fix as NMF: use `item_pop_train` (training-split only) instead of the full-dataset popularity score, for candidates with no trained NCF vector.

In [26]:
OFFSET = 1e6  # NCF-scored candidates always rank above the popularity fallback

candidate_interactions_df['ncf_hybrid_score'] = np.where(
    candidate_interactions_df['ncf_score'].notna(),
    candidate_interactions_df['ncf_score'] + OFFSET,
    candidate_interactions_df['item_pop_train']
)


In [28]:
cols_needed = [
    'session_id', 'item_id', 'split',
    'item_pop_train', 'brand_pop_train', 'cate4_pop_train', 'query_match_count',
    'cate_id_1', 'cate_id_2', 'cate_id_3', 'cate_id_4',
    'label', 'hybrid_score', 'ncf_hybrid_score'
]
test_only = candidate_interactions_df.loc[
    candidate_interactions_df['split'] == 'test', cols_needed
].copy()
test_only.to_parquet(OUTPUT_DIR / "ningjing_test_features_with_baselines.parquet", index=False)
print("Saved.", test_only.shape)

Saved. (1531525, 14)
